In [6]:
from typing import Union, Optional, Annotated, Sequence

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.constants import START


def message_send(name: Optional[str]) -> None:
    if name is None:
        print("Message sent without name")
    if name is not None:
        print(f"Message sent with {name}")

message_send(True)


Message sent with True


### Simple Graph with one node

In [7]:
from typing import Dict, TypedDict, List
from langgraph.graph import StateGraph

In [23]:
class AgentState(TypedDict):
    message: str
    values: list[str]
    name: str
    result: str

def process_values(state: AgentState) -> AgentState:
    """this function handles multiple different inputs"""
    state["result"] = f"{state['name']}, your sum = {sum(state['values'])}"
    return state

def greeting_node(state: AgentState) -> AgentState:
    """ simple node that adds a greeting message """
    state["message"] = f"Hello, {state['message']}!"
    return state

graph = StateGraph(AgentState)
graph.add_node("greeter", greeting_node)
graph.add_node("processor", process_values)

graph.add_edge("greeter", "processor")

graph.set_entry_point("greeter")
graph.set_finish_point("greeter")

app = graph.compile()
app.invoke({"message": "test", "name": "tony", "values":[1,2,3]})

{'message': 'Hello, test!',
 'values': [1, 2, 3],
 'name': 'tony',
 'result': 'tony, your sum = 6'}

### Loop Nodes

In [30]:
import random


class AgentState(TypedDict):
    message: str
    number: list[int]
    counter: int

def greeting_node(state: AgentState) -> AgentState:
    """greeting node that adds a greeting message """
    state["message"] = f"Hello, {state['message']}!"
    state["counter"] = 0
    return state

def random_node(state: AgentState) -> AgentState:
    """random node that adds a random message """
    state["number"].append(random.randint(0, 10))
    state["counter"] += 1
    return state

def should_continue(state: AgentState) -> bool:
    """function that checks if the agent should continue """
    if state["counter"] < 5:
        print(f"Agent {state['message']} should continue, counter {state['counter']}")
        return True
    else:
        return False


In [32]:
from langgraph.constants import START, END

graph = StateGraph(AgentState)
graph.add_node("greeter", greeting_node)
graph.add_node("random", random_node)

graph.add_edge(START, "greeter")
graph.add_edge("greeter", "random")
graph.add_conditional_edges("random", should_continue, {True: "random", False: END})

app = graph.compile()

In [35]:
rs = app.invoke({"message": "test", "number": []})
rs

Agent Hello, test! should continue, counter 1
Agent Hello, test! should continue, counter 2
Agent Hello, test! should continue, counter 3
Agent Hello, test! should continue, counter 4


{'message': 'Hello, test!', 'number': [9, 0, 6, 0, 9], 'counter': 5}

### 1st Agent

In [43]:
from typing import List
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
llm = ChatOpenAI(**{
    'model': "qwen-turbo",
    'api_key': os.getenv("DASHSCOPE_API_KEY"),
    'base_url': os.getenv("DASHSCOPE_API_URL"),
    'temperature': 0.1,
    "extra_body": {
        'enable_thinking': False
    }
})

class AgentState(TypedDict):
    messages: List[HumanMessage]

def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    print(response)
    return state

graph = StateGraph(AgentState)
graph.add_node("processor", process)
graph.add_edge(START, "processor")
graph.add_edge("processor", END)

app = graph.compile()


In [45]:
app.invoke({"messages": [HumanMessage(content="  上海天气怎么样？")]})

content='截至2025年4月5日，上海的天气情况如下：\n\n- **当前天气**：多云，气温约18°C，微风。\n- **今日天气**：多云转晴，气温在16°C至22°C之间，东南风3-4级。\n- **明日天气**：晴，气温略有上升，预计在19°C至25°C之间，风力减弱。\n\n建议根据天气变化适当调整着装，外出时注意防晒和补水。如果需要更详细的预报或特定区域的天气信息，请告知我具体日期或区域。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 16, 'total_tokens': 142, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-394ee74f-c3ab-9ff7-85e4-86fa4d5467fd', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--c91a12c2-5529-412d-a5f0-454354c44ada-0' usage_metadata={'input_tokens': 16, 'output_tokens': 126, 'total_tokens': 142, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


{'messages': [HumanMessage(content='上海天气怎么样？', additional_kwargs={}, response_metadata={})]}

### ReAct Model

In [70]:
from typing import Annotated, TypedDict, Sequence
from langchain_core.messages import BaseMessage

from langchain_core.tools import tool
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def add(a:int, b:int) -> int:
    """this function adds two numbers"""
    return a + b

@tool
def subtract(a:int, b:int) -> int:
    """this function subtract two numbers"""
    return a - b

@tool
def multiply(a:int, b:int) -> int:
    """this function multiply two numbers"""
    return a * b

tools = [add, subtract, multiply]

llm = ChatOpenAI(**{'model': "qwen-turbo",
    'api_key': os.getenv("DASHSCOPE_API_KEY"),
    'base_url': os.getenv("DASHSCOPE_API_URL"),
    'temperature': 0.1,
    "extra_body": {
        'enable_thinking': False
    }}).bind_tools(tools)



In [78]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode

def model_call(state: AgentState) -> AgentState:
    # print("model_cal>>>" + state)
    system_prompt = SystemMessage(content="你是一个可以使用工具的AI助手")
    response = llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> bool:
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return False
    else:
        print(f"Agent {last_message} should continue")
        return True

graph = StateGraph(AgentState)
graph.add_node("processor", model_call)

tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.add_edge(START, "processor")
graph.add_conditional_edges("processor", should_continue, {True: "tools", False: END})
graph.add_edge("tools", "processor")

app = graph.compile()


In [79]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(*message)
        else:
            message.pretty_print()

inputs = {"messages": [HumanMessage("100加101等于多少？再乘以3呢？")]}
print_stream(app.stream(inputs, stream_mode="values"))


================================ Human Message =================================

100加101等于多少？再乘以3呢？
Agent content='' additional_kwargs={'tool_calls': [{'id': 'call_c6fbb27e59ef4c798d5aff', 'function': {'arguments': '{"a": 100, "b": 101}', 'name': 'add'}, 'type': 'function', 'index': 0}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 317, 'total_tokens': 345, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}}, 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-8a5cadef-0bb7-9e90-b131-e2fe56d486eb', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='run--748c270c-eb90-481a-ac96-6a083595d511-0' tool_calls=[{'name': 'add', 'args': {'a': 100, 'b': 101}, 'id': 'call_c6fbb27e59ef4c798d5aff', 'type': 'tool_call'}] usage_metadata={'input_tokens': 317, 'output_tokens': 28, 'total_tokens': 345, 'input_token_details': {'cache_read': 256}, 'output_toke